In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# 2-3 Trees

This notebook contains the code to visualize *2-3 trees*.

This notebook presents [2-3 trees](https://en.wikipedia.org/wiki/2-3_tree).  We define these trees inductively as follows:
  - $\texttt{Nil}$ is a 2-3 tree that represents the empty set.
  - $\texttt{Two}(l, k, r)$ is a 2-3 tree provided that
      - $l$ is a 2-3 tree,
      - $k$ is a key,
      - $r$ is a 2-3 tree,
      - all keys stored in $l$ are less than k and all keys stored in $r$ are bigger than $k$, that is we have
        $$ l < k < r. $$
      - $l$ and $r$ have the same height.
      A node of the form  $\texttt{Two}(l, k, r)$ is called a *2-node*.  Except for the fact
      that there is no value, a 2-node is
      interpreted in the same way as we have interpreted the term $\texttt{Node}(k, v, l, r)$.
  - $\texttt{Three}(l, k_l, m, k_r, r)$ is a 2-3 tree provided
      - $l$, $m$, and $r$ are 2-3 trees,
      - $k_l$ and $k_r$ are keys,
      - $l < k_l < m < k_r < r$,
      - $l$, $m$, and $r$ have the same height.
      A node of the form  $\texttt{Three}(l, k_l, m, k_r, r)$ is called a *3-node*. 

In order to keep 2-3 trees balanced when inserting new keys we use a fourth constructor of the form
$\texttt{Four}(l,k_1,m_l, k_2, m_r, k_3, r)$.
A term of the form $\texttt{Four}(l,k_1,m_l, k_2, m_r, k_3, r)$ is a *2-3-4 tree* iff
  - $l$, $m_l$, $m_r$, and $r$ are 2-3 trees,
  - $k_l$, $k_m$, and $k_r$ are keys,
  - $l < k_l < m_l < k_m < m_r < k_r < r$,
  - $l$, $m_l$, $m_r$, and $r$ all have the same height.
  
Nodes of this form are called *4-nodes* and the key $k_m$ is called the *middle key*.
Trees containing 2-nodes, 3-node, and 4-nodes are called *2-3-4 trees*.

In order to keep 2-3 trees balanced when deleting keys we use a fifth constructor of the form
$\texttt{One}(t)$.
A term of the form $\texttt{One}(t)$ is a *1-2-3 tree* iff $t$ is a 2-3 tree.

The class `TwoThreeTree` is a *superclass* for constructing the nodes of *2-3-4 trees*.  It has one static variable `sNodeCount`.  This variable is used to equip all nodes with a unique identifier.  This identifier is used to draw the trees using `Graphviz`.

Every node has a uniques id `mID` that is stored as a member variable.  Furthermore, this class provides defaults for the functions `isNil`, `isTwo`, `isThree`, and `isFour`.  These functions can be used to check the type of a node.

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";

In [ ]:
class TwoThreeTree<T> {
    private static sNodeCount = 0;
    private mID: number;

    constructor() {
        TwoThreeTree.sNodeCount += 1;
        this.mID = TwoThreeTree.sNodeCount;
    }

    getID(): number {
        return this.mID;
    }

    isNil(): boolean {
        return false;
    }

    isOne(): boolean {
        return false;
    }

    isTwo(): boolean {
        return false;
    }

    isThree(): boolean {
        return false;
    }

    isFour(): boolean {
        return false;
    }

    isTree(): boolean {
        return false;
    }

    isMethod(): boolean {
        return false;
    }

    insert(k: T): this {
        return this.ins(k).restore().grow();
    }

    delete(k: T): this {
        return this.del(k).repair().shrink();
    }

    private grow(): this {
        return this;
    }

    private shrink(): this {
        return this;
    }
}
interface TwoThreeTree<T> {
    getID(): number;
    isNil(): boolean;
    isOne(): boolean;
    isTwo(): boolean;
    isThree(): boolean;
    isFour(): boolean;
    isTree(): boolean;
    isMethod(): boolean;
    insert(k: T): this;
    delete(k: T): this;
    ins(k: T): this;
    del(k: T): this;
    repair(): this;
    restore(): this;
    makeString(attributes: string[]): string;
    toDot(): Promise<string>;
    collectIDs(nodeDict: Record<number, TwoThreeTree<T>>): void;
    mChild?: TwoThreeTree<T>;
    mLeft?: TwoThreeTree<T>;
    mRight?: TwoThreeTree<T>;
    mMiddle?: TwoThreeTree<T>;
    mMiddleL?: TwoThreeTree<T>;
    mMiddleR?: TwoThreeTree<T>;
    mKey?: T;
    mKeyL?: T;
    mKeyM?: T;
    mKeyR?: T;
    mName?: string;
    mLabel?: string;
}

The function `makeString` is a helper function used to print a TwoThreeTree as a string.
  - `this` is the object that is to be rendered as a string
  - `attributes` is an array of those member variables that are used to produce the string

In [ ]:
TwoThreeTree.prototype.makeString = function<T>(this: TwoThreeTree<T>, attributes: string[]): string {
    const name = this.constructor.name;
    const values = attributes.map(attr => {
        const value = this[attr];
        return String(value);
    });
    return `${name}(${values.join(', ')})`;
};


The method $t.\texttt{toDot}()$ takes a 2-3-4 tree $t$ and returns a graph that depicts the tree $t$.

In [ ]:
TwoThreeTree.prototype.toDot = async function<T>(this: TwoThreeTree<T>): Promise<string> {
    const graphviz = await Graphviz.load();
    const dot = graphviz.graphviz();
    const nodeDict: Record<number, TwoThreeTree<T>> = {};
    this.collectIDs(nodeDict);
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.isNil() || t.isOne()) {
            dot.node(n, { label: '', shape: 'point' });
        } else if (t.isTwo()) {
            dot.node(n, { label: String(t.mKey) });
        } else if (t.isThree()) {
            dot.node(n, { label: `${t.mKeyL}|${t.mKeyR}` });
        } else if (t.isFour()) {
            dot.node(n, { label: `${t.mKeyL}|${t.mKeyM}|${t.mKeyR}` });
        } else if (t.isTree()) {
            dot.node(n, { label: String(t.mName), shape: 'triangle' });
        } else {
            throw new Error(`Unknown node ${t}`);
        }
    }

    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.isOne()) {
            dot.edge(n, String(t.mChild.getID()));
        }
        if (t.isTwo()) {
            dot.edge(n, String(t.mLeft.getID()));
            dot.edge(n, String(t.mRight.getID()));
        }
        if (t.isThree()) {
            dot.edge(n, String(t.mLeft.getID()));
            dot.edge(n, String(t.mMiddle.getID()));
            dot.edge(n, String(t.mRight.getID()));
        }
        if (t.isFour()) {
            dot.edge(n, String(t.mLeft.getID()));
            dot.edge(n, String(t.mMiddleL.getID()));
            dot.edge(n, String(t.mMiddleR.getID()));
            dot.edge(n, String(t.mRight.getID()));
        }
    }
    return dot;
};


The method $t.\mathtt{collectIDs}(d)$ takes a tree $t$ and a record $d$ and updates the record so that the following holds:
$$
d[\mathtt{id}] = n \quad \text{for every node } n \text{ in } t.
$$
Here, $\mathtt{id}$ is the unique identifier of the node $n$, i.e. $d$ associates the identifiers with the corresponding nodes.


In [ ]:
TwoThreeTree.prototype.collectIDs = function<T>(this: TwoThreeTree<T>, nodeDict: Record<number, TwoThreeTree<T>>): void {
    nodeDict[this.getID()] = this;
    if (this.isOne()) {
        this.mChild.collectIDs(nodeDict);
    } else if (this.isTwo()) {
        this.mLeft.collectIDs(nodeDict);
        this.mRight.collectIDs(nodeDict);
    } else if (this.isThree()) {
        this.mLeft.collectIDs(nodeDict);
        this.mMiddle.collectIDs(nodeDict);
        this.mRight.collectIDs(nodeDict);
    } else if (this.isFour()) {
        this.mLeft.collectIDs(nodeDict);
        this.mMiddleL.collectIDs(nodeDict);
        this.mMiddleR.collectIDs(nodeDict);
        this.mRight.collectIDs(nodeDict);
    }
};

The function $\texttt{toDotArray}(\texttt{NodeArray})$ takes an array of trees and displays them one by one.

In [ ]:
async function toDotArray<T>(NodeArray: TwoThreeTree<T>[]): Promise<string> {
    const graphviz = await Graphviz.load();
    let dot = `digraph G {\nnode [shape=record style=rounded];\n`;
    const nodeDict: Record<number, TwoThreeTree<T>> = {};
    for (const node of NodeArray) {
        node.collectIDs(nodeDict);
    }
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.isNil() || t.isOne()) {
            dot += `${n} [label="" shape=point];\n`;
        } else if (t.isTwo()) {
            dot += `${n} [label="${t.mKey}"];\n`;
        } else if (t.isThree()) {
            dot += `${n} [label="${t.mKeyL}|${t.mKeyR}"];\n`;
        } else if (t.isFour()) {
            dot += `${n} [label="${t.mKeyL}|${t.mKeyM}|${t.mKeyR}"];\n`;
        } else if (t.isTree()) {
            dot += `${n} [label="${t.mName}" shape=triangle style=solid];\n`;
        } else if (t.isMethod()) {
            dot += `${n} [label="${t.mLabel}" shape=rectangle style=dotted];\n`;
        } else {
            throw new Error(`toDotArray: Unknown node ${String(t)}`);
        }
    }
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.isOne()) {
            dot += `${n} -> ${t.mChild.getID()};\n`;
        }
        if (t.isTwo()) {
            dot += `${n} -> ${t.mLeft.getID()};\n${n} -> ${t.mRight.getID()};\n`;
        }
        if (t.isThree()) {
            dot += `${n} -> ${t.mLeft.getID()};\n${n} -> ${t.mMiddle.getID()};\n${n} -> ${t.mRight.getID()};\n`;
        }
        if (t.isFour()) {
            dot += `${n} -> ${t.mLeft.getID()};\n${n} -> ${t.mMiddleL.getID()};\n${n} -> ${t.mMiddleR.getID()};\n${n} -> ${t.mRight.getID()};\n`;
        }
    }
    dot += `}\n`;
    return dot;
}

The class `Tree` is not used in the implementation of *2-3 trees*.  It is only used for displaying abstract subtrees in equations.
It is displayed as a triangle containing the string that is stored in the member variable `mName`.

In [ ]:
class Tree<T> extends TwoThreeTree<T> {
    mName: string;
    constructor(name: string) {
        super();
        this.mName = name;
    }
    toString(): string {
        return this.mName;
    }
    isTree(): boolean {
        return true;
    }
}

The class `Method` is not used in the implementation of 2-3 trees. It is only used for displaying method calls in equations. It is displayed as a rectangle containing the string that is stored in the member variable `mLabel`.

In [ ]:
class Method<T> extends TwoThreeTree<T> {
    mLabel: string;
    constructor(label: string) {
        super();
        this.mLabel = label;
    }
    toString(): string {
        return this.mLabel;
    }
    isMethod(): boolean {
        return true;
    }
}

The class `Nil` represents an empty tree.  It has no member variables of its own.

In [ ]:
class Nil<T> extends TwoThreeTree<T> {
    constructor() {
        super();
    }
    isNil(): boolean {
        return true;
    }
    toString(): string {
        return 'Nil()';
    }
}

The class `One` represents a 1-node.  These are nodes without a key that have only a single child.

In [ ]:
class One<T> extends TwoThreeTree<T> {
    mChild: TwoThreeTree<T>;
    constructor(child: TwoThreeTree<T>) {
        super();
        this.mChild = child;
    }
    isOne(): boolean {
        return true;
    }
    toString(): string {
        return this.makeString(['mChild']);
    }
}

In [ ]:
import { display } from "tslab";

async function renderTree<T>(nodes: TwoThreeTree<T>[]): Promise<void> {
    const dot = await toDotArray(nodes);
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    display.html(svg);
}

Graphically, the node $\texttt{One}(t)$ is represented as shown below:

In [ ]:
await renderTree([new One(new Tree('t'))]);

The class `Two` represents a 2-node of the form $\texttt{Two}(l, k, r)$.  It manages three member variables:
  - `mLeft`  is the left subtree $l$,
  - `mKey`   is the key that is stored at this node,
  - `mRight` is the right subtree $r$.

In [ ]:
class Two<T> extends TwoThreeTree<T> {
    mLeft: TwoThreeTree<T>;
    mKey: T;
    mRight: TwoThreeTree<T>;
    constructor(left: TwoThreeTree<T>, key: T, right: TwoThreeTree<T>) {
        super();
        this.mLeft = left;
        this.mKey = key;
        this.mRight = right;
    }
    isTwo(): boolean {
        return true;
    }
    toString(): string {
        return this.makeString(['mLeft', 'mKey', 'mRight']);
    }
}

Graphically, the node $\texttt{Two}(l, k, r)$ is represented as shown below:

In [ ]:
await renderTree([new Two(new Tree('l'), 'k', new Tree('r'))]);

The class `Three` represents a 3-node of the form $\texttt{Three}(l, k_L, m, k_R, r)$.  It manages 5 member variables:
  - `mLeft`   is the left subtree $l$,
  - `mKeyL`   is the left key $k_L$,
  - `mMiddle` is the middle subtree $m$,
  - `mKeyR`   is the right key $k_r$,
  - `mRight`  is the right subtree.

In [ ]:
class Three<T> extends TwoThreeTree<T> {
    mLeft: TwoThreeTree<T>;
    mKeyL: T;
    mMiddle: TwoThreeTree<T>;
    mKeyR: T;
    mRight: TwoThreeTree<T>;
    constructor(left: TwoThreeTree<T>, keyL: T, middle: TwoThreeTree<T>, keyR: T, right: TwoThreeTree<T>) {
        super();
        this.mLeft = left;
        this.mKeyL = keyL;
        this.mMiddle = middle;
        this.mKeyR = keyR;
        this.mRight = right;
    }
    toString(): string {
        return this.makeString(['mLeft', 'mKeyL', 'mMiddle', 'mKeyR', 'mRight']);
    }
    isThree(): boolean {
        return true;
    }
}

Graphically, the node $\texttt{Three}(l, k_L, m, k_R, r)$ is represented as shown below:

In [ ]:
await renderTree([new Three(new Tree('l'), 'kL', new Tree('m'), 'kR', new Tree('r'))]);

The class `Four` represents a 4-node.  It manages 7 member variables:
  - `mLeft`    is the left subtree $l$,
  - `mKeyL`    is the left key $k_L$,
  - `mMiddleL` is the middle left subtree $m_L$,
  - `mKeyM`    is the middle key,
  - `mMiddleR` is the middle right subtree $m_R$,
  - `mKeyR`    is the right key $k_r$,
  - `mRight`   is the right subtree.

In [ ]:
class Four<T> extends TwoThreeTree<T> {
    mLeft: TwoThreeTree<T>;
    mKeyL: T;
    mMiddleL: TwoThreeTree<T>;
    mKeyM: T;
    mMiddleR: TwoThreeTree<T>;
    mKeyR: T;
    mRight: TwoThreeTree<T>;
    constructor(
        l: TwoThreeTree<T>, kl: T, ml: TwoThreeTree<T>,
        km: T, mr: TwoThreeTree<T>, kr: T, r: TwoThreeTree<T>
    ) {
        super();
        this.mLeft = l;
        this.mKeyL = kl;
        this.mMiddleL = ml;
        this.mKeyM = km;
        this.mMiddleR = mr;
        this.mKeyR = kr;
        this.mRight = r;
    }
    toString(): string {
        return this.makeString([
            'mLeft', 'mKeyL', 'mMiddleL', 'mKeyM', 'mMiddleR', 'mKeyR', 'mRight'
        ]);
    }
    isFour(): boolean {
        return true;
    }
}

Graphically, the node $\texttt{Four}(l, k_L, m_L, k_M, m_R, k_R, r)$ is represented as shown below:

In [ ]:
await renderTree([new Four(new Tree('l'), 'kL', new Tree('mL'), 'kM', new Tree('mR'), 'kR', new Tree('r'))]);

## Methods of the Class `Nil`

Insertings a key $k$ into an empty node returns a 2-node with two empty subtrees.

In [ ]:
await renderTree([new Nil(), new Method('insert(k)'), new Two(new Nil(), 'k', new Nil())]);

## Methods of the Class Two

The method `extract` returns the member variables stored in a 2-node.  This is usefull to shorten the code since when we use this method, we don't have to prefix all variable names with `self.`.

The method $t.\texttt{ins}(k)$ takes a 2-3 tree $t$ and and a key $k$ and inserts the key $k$ into $t$.
It returns a 2-3-4 tree that has at most one 4-node, which has to the root
node.  The function $\texttt{ins}$ is recursive and uses the function $\texttt{restore}$ defined below. 

The most important invariant satisfied by the method call $t.\texttt{ins}(k)$ is the fact that the tree
$t.\texttt{ins}(k)$ has the same height as the tree $t$.

The different cases that need to be handled by `ins` are shown graphically below:

In [ ]:
await renderTree([new Two(new Tree('l'), 'k', new Tree('r')), new Method('.ins(k)'), new Two(new Tree('l'), 'k', new Tree('r'))]);

In [ ]:
await renderTree([new Method('[k1 < k2:]'), new Two(new Nil(), 'k1', new Nil()), new Method('.ins(k2)'), new Three(new Nil(), 'k1', new Nil(), 'k2', new Nil())]);

In [ ]:
await renderTree([new Method('[k2 < k1:]'), new Two(new Nil(), 'k1', new Nil()), new Method('.ins(k2)'), new Three(new Nil(), 'k2', new Nil(), 'k1', new Nil())]);

In [ ]:
await renderTree([new Method('[k1 < k2:]'), new Two(new Tree('l'), 'k1', new Tree('r')), new Method('.ins(k2)'), new Two(new Tree('l'), 'k1', new Tree('[r.ins(k2)]'))]);

In [ ]:
await renderTree([new Method('[:k2 < k1:]'), new Two(new Tree('l'), 'k1', new Tree('r')), new Method('.ins(k2)'), new Two(new Tree('[l.ins(k2)]'), 'k1', new Tree('r'))]);

The function call $t.\texttt{restore}()$ takes a 2-3-4 tree $t$ that has at most one 4-node. This 4-node has to be a child
of the root.  It returns a 2-3-4 tree that has at most one 4-node.  This 4-node has to be the root node.
Graphically, it is specified as shown below.

In [ ]:
await renderTree([new Two(new Four(new Tree('l1'), 'kl', new Tree('ml'), 'km', new Tree('mr'), 'kr', new Tree('r1')), 'k', new Tree('r')), 
                  new Method('.restore()'), 
                  new Three(new Two(new Tree('l1'), 'kl', new Tree('ml')), 'km', new Two(new Tree('mr'), 'kr', new Tree('r1')), 'k', new Tree('r'))]);

In [ ]:
await renderTree([new Two(new Tree('l'), 'k', new Four(new Tree('l1'), 'kl', new Tree('ml'), 'km', new Tree('mr'), 'kr', new Tree('r1'))), 
                  new Method('.restore()'), 
                  new Three(new Tree('l'), 'k', new Two(new Tree('l1'), 'kl', new Tree('ml')), 'km', new Two(new Tree('mr'), 'kr', new Tree('r1')))]);

## Methods of the Class `Three`

The function call $t.\texttt{restore}()$ takes a 2-3-4 tree $t$ that has at most one 4-node. This 4-node has to be a child
of the root.  It returns a 2-3-4 tree that has at most one 4-node.  This 4-node has to be the root node.

The most important invariant satisfied by the method call $t.\texttt{ins}(k)$ is the fact that the tree
$t.\texttt{ins}(k)$ has the same height as the tree $t$.

The different cases that need to be handled by `ins` are shown graphically below:

In [ ]:
await renderTree([new Three(new Four(new Tree('l1'), 'k1', new Tree('ml'), 'k2', new Tree('mr'), 'k3', new Tree('r1')), 'kl', new Tree('m'), 'kr', new Tree('r')), 
                  new Method('.restore()'), 
                  new Four(new Two(new Tree('l1'), 'k1', new Tree('ml')), 'k2', new Two(new Tree('mr'), 'k3', new Tree('r1')), 'kl', new Tree('m'), 'kr', new Tree('r'))]);

In [ ]:
await renderTree([new Three(new Tree('l'), 'kl', new Four(new Tree('l1'), 'k1', new Tree('ml'), 'k2', new Tree('mr'), 'k3', new Tree('r1')), 'kr', new Tree('r')), 
                  new Method('.restore()'), 
                  new Four(new Tree('l'), 'kl', new Two(new Tree('l1'), 'k1', new Tree('ml')), 'k2', new Two(new Tree('mr'), 'k3', new Tree('r1')), 'kr', new Tree('r'))]);

In [ ]:
await renderTree([new Three(new Tree('l'), 'kl', new Tree('m'), 'kr', new Four(new Tree('l1'), 'k1', new Tree('ml'), 'k2', new Tree('mr'), 'k3', new Tree('r1'))), 
                  new Method('.restore()'), 
                  new Four(new Tree('l'), 'kl', new Tree('m'), 'kr', new Two(new Tree('l1'), 'k1', new Tree('ml')), 'k2', new Two(new Tree('mr'), 'k3', new Tree('r1')))]);

## Methods of the Class `Four`

In [ ]:
await renderTree([new Four(new Tree('l'), 'kl', new Tree('ml'), 'km', new Tree('mr'), 'kr', new Tree('r')), 
                  new Method('.grow()'), 
                  new Two(new Two(new Tree('l'), 'kl', new Tree('ml')), 'km', new Two(new Tree('mr'), 'kr', new Tree('r')))]);

## Deletion

The method $t.\texttt{delete}(k)$ is specified as follows:
 - $t.\texttt{delete}(x) = t.\texttt{del}(x).\texttt{repair}().\texttt{shrink}()$

When $t.\texttt{del}(e)$ is called, $t$ is a 1-2-3 tree containing at most one 1-node.  This 1-node must occur at the root of the tree.  The expression $t.\texttt{del}(e)$ returns the 1-2-3 tree that results from removing the element $e$ from the tree $t$.

The important invariant maintained by `del` is that the trees $t$ and $t.\texttt{del}(k)$ have the same height.

The method $t.\texttt{repair}()$ takes a 1-2-3 tree $t$ that contains at most a single 1-node. 
If the tree $t$ has size $1$, then this 1-node may occur at the root.  Otherwise, the 1-node has to be a child of the root.
The method returns a 1-2-3 tree of the same size as $t$ where the 1-node can only occur at the root.

In [ ]:
await renderTree([new Two(new One(new Tree('L1')), 'k', new Two(new Tree('M1'), 'k1', new Tree('R1'))), 
                  new Method('.repair()'), 
                  new One(new Three(new Tree('L1'), 'k', new Tree('M1'), 'k1', new Tree('R1')))]);

In [ ]:
await renderTree([new Two(new Two(new Tree('L1'), 'k1', new Tree('M1')), 'k', new One(new Tree('R1'))), 
                  new Method('.repair()'), 
                  new One(new Three(new Tree('L1'), 'k1', new Tree('M1'), 'k', new Tree('R1')))]);

In [ ]:
await renderTree([new Two(new One(new Tree('A')), 'k', new Three(new Tree('B'), 'k1', new Tree('C'), 'k2', new Tree('D'))), 
                  new Method('.repair()'), 
                  new Two(new Two(new Tree('A'), 'k', new Tree('B')), 'k1', new Two(new Tree('C'), 'k2', new Tree('D')))]);

In [ ]:
await renderTree([new Two(new Three(new Tree('A'), 'k1', new Tree('B'), 'k2', new Tree('C')), 'k', new One(new Tree('D'))), 
                  new Method('.repair()'), 
                  new Two(new Two(new Tree('A'), 'k1', new Tree('B')), 'k2', new Two(new Tree('C'), 'k', new Tree('D')))]);

In [ ]:
await renderTree([new Three(new One(new Tree('A')), 'kL', new Two(new Tree('B'), 'k', new Tree('C')), 'kR', new Tree('R')), 
                  new Method('.repair()'), 
                  new Two(new Three(new Tree('A'), 'kL', new Tree('B'), 'k', new Tree('C')), 'kR', new Tree('R'))]);

In [ ]:
await renderTree([new Three(new Two(new Tree('A'), 'k', new Tree('B')), 'kL', new One(new Tree('C')), 'kR', new Tree('R')), 
                  new Method('.repair()'), 
                  new Two(new Three(new Tree('A'), 'k', new Tree('B'), 'kL', new Tree('C')), 'kR', new Tree('R'))]);

In [ ]:
await renderTree([new Three(new Tree('L'), 'kL', new Two(new Tree('A'), 'k', new Tree('B')), 'kR', new One(new Tree('C'))), 
                  new Method('.repair()'), 
                  new Two(new Tree('L'), 'kL', new Three(new Tree('A'), 'k', new Tree('B'), 'kR', new Tree('C')))]);

In [ ]:
await renderTree([new Three(new One(new Tree('A')), 'kL', new Three(new Tree('B'), 'k1', new Tree('C'), 'k2', new Tree('D')), 'kR', new Tree('R')), 
                  new Method('.repair()'), 
                  new Three(new Two(new Tree('A'), 'kL', new Tree('B')), 'k1', new Two(new Tree('C'), 'k2', new Tree('D')), 'kR', new Tree('R'))]);

In [ ]:
await renderTree([new Three(new Three(new Tree('A'), 'k1', new Tree('B'), 'k2', new Tree('C')), 'kL', new One(new Tree('D')), 'kR', new Tree('R')), 
                  new Method('.repair()'), 
                  new Three(new Two(new Tree('A'), 'k1', new Tree('B')), 'k2', new Two(new Tree('C'), 'kL', new Tree('D')), 'kR', new Tree('R'))]);

In [ ]:
await renderTree([new Three(new Tree('L'), 'kL', new Three(new Tree('A'), 'k1', new Tree('B'), 'k2', new Tree('C')), 'kR', new One(new Tree('D'))), 
                  new Method('.repair()'), 
                  new Three(new Tree('L'), 'kL', new Two(new Tree('A'), 'k1', new Tree('B')), 'k2', new Two(new Tree('C'), 'kR', new Tree('D')))]);

Finally, we specify the method `shrink`.  The only class where the implementation of `shrink` is nontrivial is the class `One`.

In [ ]:
await renderTree([new One(new Tree('A')), new Method('.shrink()'), new Tree('A')]);